# NBA 3.1 - 09 Claim comorbidity / Dx panel features (per claims cohort)

Upgrade of NBA 3.0 `03_Feature_Engineering/03_HCP_Level_Features/09_claim_comorbidity`. Spec: `Feature Analysis/92_competitor_comorbidity_nba31_readiness_plan.md` Phase 1 (CM1-CM6) and `02_comorbidity_dx_features_nba31_changes.md` section 2. Plain-language summary: `93_competitor_comorbidity_plan_in_simple_words.md`.

**What changes vs NBA 3.0**

| NBA 3.0 (39 columns) | NBA 3.1 (14 columns) |
| --- | --- |
| `Dx_09` with `FIN_DIAG_CD` groups `KD` / `DB-KD` / `CV-KD` (CKD only, stage 3a/3b missing) | `dx_input_claims_all_events_features_input` (NBA 3.1 Dx event list, ICD-10 codes) grouped by Kerendia indication (CM1) |
| Monthly counts, rolling averages 1/3/6/12M, running totals, 1/3/6/12M shares | Distinct patients in fixed 3 / 12 / 24-month windows (CM4, CM5) |
| Hard-coded `date_start`, one row per HCP x month | One row per HCP x claims cohort, claims up to the end of the cohort's `lookback_end_date` month |

**Columns** (`{stem}` suffix `_IN_LAST_365_DAYS_DX` unless noted)

| # | Column | Plan ref |
| --- | --- | --- |
| 1-8 | `NUM_` / `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_{T2D_CKD, HF_LVEF_GE_40_PROXY, CKD_AND_HF, ANY_KERENDIA_INDICATION}` | CM2 |
| 9 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_T1D_CKD` (count only) | CM2 |
| 10-11 | `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_CKD_STAGE_3`, `..._CKD_STAGE_4` | CM3 |
| 12-13 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_{DIABETIC_CKD, CKD}_IN_LAST_730_DAYS_DX` | CM4 |
| 14 | `LOGRATIO_PATIENTS_ANY_KERENDIA_INDICATION_3M` | CM6 |

**Business rules**
* Dx panel = distinct patients with any Dx event-list claim where the HCP is the referring doctor, else the rendering doctor, in the window.
* Patient flags come from claims by **any** provider in the same window (12M flags for the 365-day and log-ratio features, 24M flags for the 730-day features).
* T2D `E11*`, T1D `E10*`, HF `I50*`. CKD (non-ESRD) `N18.1-N18.5` (incl. `N18.30/31/32`), `N18.9`, `E11.21/22/29`, `E10.21/22/29`, `I12.9`, `I13.0`, `I13.10`. ESRD (excluded) `N18.6`, `Z99.2`, `I12.0`, `I13.11`, `I13.2`. Stage 3 `N18.3`, `N18.30-32`. Stage 4 `N18.4`.
* HF LVEF >= 40% proxy = an `I50.3x` code and no `I50.2x` code in the window. **Pending Medical sign-off (V3).**
* T2D+CKD / T1D+CKD / CKD+HF / diabetic CKD all require no ESRD. Any indication = T2D+CKD or T1D+CKD or HF proxy.
* Log-ratio = `ln((n_3M/3 + 0.5) / ((n_12M - n_3M)/9 + 0.5))`, n = panel patients with any indication (formula from doc 02 section 2.3; note that a patient seen in both periods is counted only in the 3M part).
* Null policy: `NUM_*` = 0 when nothing qualifies; `PCT_*` and the log-ratio are empty when the HCP has no Dx panel in the window.

Inputs: 3.1 volume only. Output: `hcp_level_model/dev_pipeline/notebook_output/hcp_features/claim_comorbidity` (Delta), key `HCP_COHORT_ID`. Widget `TEST_HCP_SAMPLE` > 0 runs on a sample and writes to `claim_comorbidity_test`.

## Phase 0 validation status (plan 92, Section 3), checked 2026-11-17

**Score: 6 of 10 met, 3 partly met, 1 not met.** Checks that matter for this notebook are in **bold**.

| # | Validation | Result | Status |
| --- | --- | --- | --- |
| V1 | Indication coverage in LAAD | Share of NBRx with an indication group, monthly 2023-06 to 2026-09: SGLT2 91.5-93.4%, Entresto 100%, steroidal MRA 44.1-49.3%. Stable over time | Met (MRA coverage low but stable) |
| **V2** | **CKD stage 3 codes present** | **Dx event list (this notebook's source): 11,685,734 stage-3 patients vs 26,200 in NBA 3.0. `Dx_03`: only 25,167, and 3,144,676 stage-3 patients have no `Dx_03` CKD claim at all** | **Met for this notebook**; not met for `Dx_03` (affects `11_adoption_features`) |
| **V3** | **HF LVEF >= 40% proxy sign-off** | **Medical review not done** | **Not met.** The 2 `HF_LVEF_GE_40_PROXY` columns and `ANY_KERENDIA_INDICATION` (uses the proxy) are provisional |
| V4 | CKD indication groups in LRx | V7 groups: CKD, T2D + CKD, T2D + CKD + HF, CKD + HF (no CVD groups in V7) | Met |
| V5 | Farxiga brand vs generic labels | Two labels only. Generic NBRx exists from 2023-11-03 (about 9-15k a month) and rises from Jan 2026 (43k), not from 2026-04-06 | Partly met |
| V6 | Entresto generic | `Rx_01_new`: SACUBITRIL/VALSARTAN from 2024-11-27. LRx: generic from 2025-01-31, ramp Jul-Aug 2025 (541 to 13,162 NBRx) | Met (exact LOE day still Q6) |
| V7 | GLP-1 in LRx | 12M NBRx 7.01M; 5.29M in T2D groups, 0.74M in HF groups | Met |
| V8 | Parity vs NBA 3.0 | Proxy only: 12M Jardiance / Farxiga NBRx vs the NBA 3.0 monthly output, correlation 0.71-0.93 by cohort; 3.1 totals about 20% higher (V7 vs V6) | Partly met (not a full 3.0 rebuild) |
| V9 | Rows after each event | Entresto LOE: 14 cohorts, 4,126,178 rows, 86,065 writers. Farxiga LOE: 5 cohorts, 1,473,635 rows, 32,731 writers | Met (c28 target month Oct 2026 is incomplete in LRx) |
| V10 | Entresto prescriber specialty mix | Brand writers 2023-24: PCP 49.1%, NP/PA 31.3%, CARD 16.1%, NEPH 2.6%. Kerendia writers: PCP 42.1%, NP/PA 29.6%, NEPH 19.0%, CARD 3.4% | Met (answered) |

**For this notebook:** V2 passes on its source. The build goes ahead; the HF-proxy columns stay provisional until Medical signs off V3.

In [0]:
import os, sys, json, builtins, importlib
from functools import reduce
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0 = full universe)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))

# Shared constants: hcp_features_config.py in this folder (paths, Product Basket, code sets)
sys.path.insert(0, os.getcwd())
import hcp_features_config as cfg
importlib.reload(cfg)

VOL31 = cfg.VOL31
HCP_UNIVERSE_PATH = cfg.HCP_UNIVERSE_PATH          # 294,727 HCPs (C1-C4, no call-only HCPs)
COHORTS_DIR = cfg.COHORTS_DIR
QUERY_OUTPUT_31 = cfg.QUERY_OUTPUT_31
HCP_FEATURES_OUT_31 = cfg.HCP_FEATURES_OUT_31
DX_INPUT = cfg.DX_EVENTS_PATH   # Dx event list, ICD-10 codes, 2021-01-02 onwards
SUFFIX = "_test" if TEST_HCP_SAMPLE else ""
OUTPUT_PATH = f"{HCP_FEATURES_OUT_31}/claim_comorbidity{SUFFIX}"
SCRATCH = f"{HCP_FEATURES_OUT_31}/_scratch/claim_comorbidity{SUFFIX}"
assert all("kerendia_nba_3_1_volume" in p for p in [OUTPUT_PATH, SCRATCH]), "never write outside the 3.1 volume"

# ICD-10 building blocks (doc 02 section 2.0): from the latest dated Dx Product Basket (cfg.dx_code_sets)
DX_BASKET_PATH = cfg.product_mapping_path("dx")
_DX_SETS = cfg.dx_code_sets()
CKD_CODES  = _DX_SETS["CKD"]
ESRD_CODES = _DX_SETS["ESRD"]
ST3_CODES  = _DX_SETS["ST3"]
ST4_CODES  = _DX_SETS["ST4"]
print(f"Dx basket {os.path.basename(DX_BASKET_PATH)}: CKD {len(CKD_CODES)}, ESRD {len(ESRD_CODES)}, "
      f"stage 3 {len(ST3_CODES)}, stage 4 {len(ST4_CODES)} codes")

_code = F.upper(F.trim(F.col("DIAGNOSIS_CODE")))
def _starts(*prefixes):
    return reduce(lambda a, b: a | b, [_code.startswith(p) for p in prefixes])
CODE_FLAGS = {
    "T2D": _starts(*cfg.DX_PREFIXES["T2D"]), "T1D": _starts(*cfg.DX_PREFIXES["T1D"]),
    "CKD": _code.isin(CKD_CODES), "ESRD": _code.isin(ESRD_CODES),
    "ST3": _code.isin(ST3_CODES), "ST4": _code.isin(ST4_CODES),
    "HF": _starts(*cfg.DX_PREFIXES["HF"]), "HF_SYS": _starts(*cfg.DX_PREFIXES["HF_SYS"]),
    "HF_DIA": _starts(*cfg.DX_PREFIXES["HF_DIA"]),
}

P = "OF_PATIENTS_IN_HCP_DX_PANEL_WITH"
FEATURE_COLS = (
    [f"{k}_{P}_{s}_IN_LAST_365_DAYS_DX" for s in ["T2D_CKD", "HF_LVEF_GE_40_PROXY", "CKD_AND_HF", "ANY_KERENDIA_INDICATION"] for k in ["NUM", "PCT"]]
    + [f"NUM_{P}_T1D_CKD_IN_LAST_365_DAYS_DX",
       f"PCT_{P}_CKD_STAGE_3_IN_LAST_365_DAYS_DX", f"PCT_{P}_CKD_STAGE_4_IN_LAST_365_DAYS_DX",
       f"NUM_{P}_DIABETIC_CKD_IN_LAST_730_DAYS_DX", f"NUM_{P}_CKD_IN_LAST_730_DAYS_DX",
       "LOGRATIO_PATIENTS_ANY_KERENDIA_INDICATION_3M"])
assert len(FEATURE_COLS) == 14
print(f"TEST_HCP_SAMPLE={TEST_HCP_SAMPLE} -> {OUTPUT_PATH}")

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims", meta["feature_family"]
assert len(cohorts) == meta["cohort_count"] and (cohorts["offset_days"] == 0).all()

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string")).distinct()
if TEST_HCP_SAMPLE:
    hcp_universe = hcp_universe.orderBy(F.xxhash64("PROVIDER_ID")).limit(TEST_HCP_SAMPLE)
hcp_universe = hcp_universe.cache()
n_hcp = hcp_universe.count()

END_MONTHS = [pd.to_datetime(d).date() for d in cohorts["lookback_end_date"]]
FIRST_MONTH = (pd.to_datetime(min(END_MONTHS)) - pd.DateOffset(months=23)).date()   # 24-month window of the first cohort
LAST_MONTH = max(END_MONTHS)
print(f"{len(cohorts)} claims cohorts, lookback end {min(END_MONTHS)}..{LAST_MONTH} | Dx months used {FIRST_MONTH}..{LAST_MONTH} | HCPs {n_hcp:,}")

In [0]:
dx = (spark.read.format("delta").load(DX_INPUT)
      .withColumn("HCP", F.coalesce("PROVIDER_REFERRING_ID", "PROVIDER_RENDERING_ID"))   # referring, else rendering
      .withColumn("M", F.trunc("SVC_DT", "month"))
      .filter(F.col("PATIENT_ID").isNotNull()
              & (F.col("M") >= F.lit(str(FIRST_MONTH)).cast("date"))
              & (F.col("M") <= F.lit(str(LAST_MONTH)).cast("date"))))

# HCP x patient x month: the Dx panel (universe HCPs only)
panel_m = (dx.join(F.broadcast(hcp_universe.withColumnRenamed("PROVIDER_ID", "HCP")), "HCP", "left_semi")
             .select("HCP", "PATIENT_ID", "M").distinct())
panel_m.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SCRATCH}/panel_m")
panel_m = spark.read.format("delta").load(f"{SCRATCH}/panel_m")

# Patient x month flags from claims by ANY provider (only patients that appear in a universe panel)
pm = (dx.join(panel_m.select("PATIENT_ID").distinct(), "PATIENT_ID", "left_semi")
        .groupBy("PATIENT_ID", "M").agg(*[F.max(v.cast("int")).alias(k) for k, v in CODE_FLAGS.items()]))
pm.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SCRATCH}/patient_month_flags")
pm = spark.read.format("delta").load(f"{SCRATCH}/patient_month_flags")
print(f"panel rows {panel_m.count():,} | patient-month rows {pm.count():,}")

In [0]:
def _window(df, end_m, n_months):
    end = F.lit(str(end_m)).cast("date")
    return df.filter((F.col("M") >= F.add_months(end, -(n_months - 1))) & (F.col("M") <= end))

def _patient_flags(end_m, n_months):
    p = _window(pm, end_m, n_months).groupBy("PATIENT_ID").agg(*[F.max(k).alias(k) for k in CODE_FLAGS])
    no_esrd = F.col("ESRD") == 0
    t2d_ckd = (F.col("T2D") == 1) & (F.col("CKD") == 1) & no_esrd
    t1d_ckd = (F.col("T1D") == 1) & (F.col("CKD") == 1) & no_esrd
    hf_proxy = (F.col("HF_DIA") == 1) & (F.col("HF_SYS") == 0)
    return p.select("PATIENT_ID",
        t2d_ckd.cast("int").alias("T2D_CKD"), t1d_ckd.cast("int").alias("T1D_CKD"),
        hf_proxy.cast("int").alias("HF_PROXY"),
        ((F.col("CKD") == 1) & (F.col("HF") == 1) & no_esrd).cast("int").alias("CKD_AND_HF"),
        (t2d_ckd | t1d_ckd | hf_proxy).cast("int").alias("ANY_IND"),
        (((F.col("T2D") == 1) | (F.col("T1D") == 1)) & (F.col("CKD") == 1) & no_esrd).cast("int").alias("DIAB_CKD"),
        ((F.col("CKD") == 1) & no_esrd).cast("int").alias("CKD_NE"),
        F.col("ST3"), F.col("ST4"))

def _panel(end_m, n_months, flags):
    return (_window(panel_m, end_m, n_months).select("HCP", "PATIENT_ID").distinct()
            .join(flags, "PATIENT_ID", "left").fillna(0))

parts = []
for _, c in cohorts.iterrows():
    end_m = pd.to_datetime(c.lookback_end_date).date()
    f12, f24 = _patient_flags(end_m, 12), _patient_flags(end_m, 24)
    a12 = _panel(end_m, 12, f12).groupBy("HCP").agg(
        F.count("*").alias("PANEL_12"),
        *[F.sum(k).alias(k) for k in ["T2D_CKD", "T1D_CKD", "HF_PROXY", "CKD_AND_HF", "ANY_IND", "ST3", "ST4"]])
    a3 = _panel(end_m, 3, f12.select("PATIENT_ID", "ANY_IND")).groupBy("HCP").agg(F.sum("ANY_IND").alias("ANY_IND_3"))
    a24 = _panel(end_m, 24, f24).groupBy("HCP").agg(F.sum("DIAB_CKD").alias("DIAB_CKD_24"), F.sum("CKD_NE").alias("CKD_24"))
    parts.append(a12.join(a3, "HCP", "left").join(a24, "HCP", "full")
                 .withColumn("cohort_id", F.lit(c.cohort_id)).withColumn("SVC_DT_M", F.lit(str(end_m)).cast("date")))
long = reduce(lambda a, b: a.unionByName(b), parts).withColumnRenamed("HCP", "PROVIDER_ID")

def _pct(num):
    return F.when(F.col("PANEL_12") > 0, F.coalesce(F.col(num), F.lit(0)) / F.col("PANEL_12"))

n = lambda c: F.coalesce(F.col(c), F.lit(0))
feat = long.select(
    "PROVIDER_ID", "cohort_id",
    n("T2D_CKD").alias(FEATURE_COLS[0]),     _pct("T2D_CKD").alias(FEATURE_COLS[1]),
    n("HF_PROXY").alias(FEATURE_COLS[2]),    _pct("HF_PROXY").alias(FEATURE_COLS[3]),
    n("CKD_AND_HF").alias(FEATURE_COLS[4]),  _pct("CKD_AND_HF").alias(FEATURE_COLS[5]),
    n("ANY_IND").alias(FEATURE_COLS[6]),     _pct("ANY_IND").alias(FEATURE_COLS[7]),
    n("T1D_CKD").alias(FEATURE_COLS[8]),
    _pct("ST3").alias(FEATURE_COLS[9]),      _pct("ST4").alias(FEATURE_COLS[10]),
    n("DIAB_CKD_24").alias(FEATURE_COLS[11]), n("CKD_24").alias(FEATURE_COLS[12]),
    F.when(F.col("PANEL_12") > 0,
           F.log((n("ANY_IND_3") / 3 + 0.5) / ((n("ANY_IND") - n("ANY_IND_3")) / 9 + 0.5))).alias(FEATURE_COLS[13]))

# Every universe HCP x every cohort. NUM_* = 0 when nothing qualifies; PCT_* / log-ratio stay empty without a panel
coh = spark.createDataFrame(cohorts[["cohort_id", "lookback_end_date"]].astype(str)) \
           .withColumn("SVC_DT_M", F.to_date("lookback_end_date")).drop("lookback_end_date")
num_cols = [c for c in FEATURE_COLS if c.startswith("NUM_")]
claim_comorbidity = (hcp_universe.crossJoin(F.broadcast(coh))
    .join(feat, ["PROVIDER_ID", "cohort_id"], "left")
    .fillna(0, subset=num_cols)
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID",
            *[F.col(c).cast("int").alias(c) if c.startswith("NUM_") else F.col(c).cast("double").alias(c) for c in FEATURE_COLS],
            "PULL_DT"))

os.makedirs(HCP_FEATURES_OUT_31, exist_ok=True)
claim_comorbidity.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH)
out = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {out.count():,} rows, {len(out.columns)} columns -> {OUTPUT_PATH}")

In [0]:
n_rows = out.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert out.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"
assert out.join(hcp_universe, "PROVIDER_ID", "left_anti").count() == 0

pct_cols = [c for c in FEATURE_COLS if c.startswith("PCT_")]
bad_pct = out.filter(" OR ".join(f"({c} < 0 OR {c} > 1)" for c in pct_cols)).count()
bad_num = out.filter(" OR ".join(f"({c} < 0 OR {c} IS NULL)" for c in num_cols)).count()
ANY = FEATURE_COLS[6]
bad_any = out.filter(f"{ANY} < greatest({FEATURE_COLS[0]}, {FEATURE_COLS[2]}, {FEATURE_COLS[8]})").count()
print(f"rows {n_rows:,} | PCT outside [0,1]: {bad_pct} | NUM null/negative: {bad_num} | ANY < a component: {bad_any}")
assert bad_pct == 0 and bad_num == 0 and bad_any == 0

# Independent check on 200 HCPs for the last cohort: recompute from raw claims with SVC_DT date bounds
last = cohorts.iloc[-1]
end = F.last_day(F.lit(str(last.lookback_end_date)[:10]).cast("date"))
start12 = F.add_months(F.trunc(end, "month"), -11)
chk_hcps = out.filter(F.col("cohort_id") == last.cohort_id).orderBy(F.xxhash64("PROVIDER_ID")).select("PROVIDER_ID").limit(200)
raw = (spark.read.format("delta").load(DX_INPUT).filter((F.col("SVC_DT") >= start12) & (F.col("SVC_DT") <= end))
       .withColumn("HCP", F.coalesce("PROVIDER_REFERRING_ID", "PROVIDER_RENDERING_ID")))
panel_chk = raw.join(chk_hcps.withColumnRenamed("PROVIDER_ID", "HCP"), "HCP", "left_semi").select("HCP", "PATIENT_ID").distinct()
flags_chk = (raw.join(panel_chk.select("PATIENT_ID").distinct(), "PATIENT_ID", "left_semi").groupBy("PATIENT_ID")
             .agg(F.max(CODE_FLAGS["T2D"].cast("int")).alias("t2d"), F.max(CODE_FLAGS["CKD"].cast("int")).alias("ckd"),
                  F.max(CODE_FLAGS["ESRD"].cast("int")).alias("esrd")))
exp = (panel_chk.join(flags_chk, "PATIENT_ID").groupBy("HCP")
       .agg(F.sum(((F.col("t2d") == 1) & (F.col("ckd") == 1) & (F.col("esrd") == 0)).cast("int")).alias("exp_t2d_ckd"))
       .withColumnRenamed("HCP", "PROVIDER_ID"))
cmp = (out.filter(F.col("cohort_id") == last.cohort_id).join(chk_hcps, "PROVIDER_ID")
          .join(exp, "PROVIDER_ID", "left").fillna(0, subset=["exp_t2d_ckd"]))
mism = cmp.filter(F.col(FEATURE_COLS[0]) != F.col("exp_t2d_ckd")).count()
print(f"{last.cohort_id}: T2D+CKD count mismatches vs direct recompute (200 HCPs) = {mism}")
assert mism == 0

In [0]:
last_df = out.filter(F.col("cohort_id") == cohorts.cohort_id.iloc[-1])
stats = last_df.agg(*[F.count(F.when(F.col(c) > 0, 1)).alias(f"nz_{c}") for c in FEATURE_COLS],
                    *[F.percentile_approx(c, 0.5).alias(f"med_{c}") for c in FEATURE_COLS]).collect()[0]
display(pd.DataFrame({"feature": FEATURE_COLS,
                      "pct_hcps_non_zero": [builtins.round(100 * stats[f"nz_{c}"] / n_hcp, 1) for c in FEATURE_COLS],
                      "median": [stats[f"med_{c}"] for c in FEATURE_COLS]}))

# V2: CKD stage 3 patients (N18.3, N18.30-32) in the 12 months of the last cohort; NBA 3.0 Dx_09 had 26,200 (N18.3 only)
st3 = pm.filter((F.col("ST3") == 1) & (F.col("M") >= F.add_months(F.lit(str(LAST_MONTH)).cast("date"), -11))) \
        .select("PATIENT_ID").distinct().count()
print(f"V2: stage 3 patients in the last 12 months (universe panels): {st3:,}")

In [0]:
# Output table: one row per HCP x claims cohort (key HCP_COHORT_ID)
# Reads the saved output, so this cell and the next only need 'Paths and parameters' and 'Load claims cohorts' run first
out = spark.read.format("delta").load(OUTPUT_PATH)
FEATURE_COLS = [c for c in out.columns if c not in {"PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", "PULL_DT"}]
display(out.select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", *FEATURE_COLS))

In [0]:
# Fill rate per feature: % of rows with a value (not null) and % with a non-zero value,
# over all cohorts and for the latest cohort
latest = cohorts.cohort_id.iloc[-1]

def _fill(df):
    n = df.count()
    r = df.agg(*[F.count(c).alias(f"nn_{c}") for c in FEATURE_COLS],
               *[F.count(F.when(F.col(c) != 0, 1)).alias(f"nz_{c}") for c in FEATURE_COLS]).collect()[0]
    return ([builtins.round(100 * r[f"nn_{c}"] / n, 2) for c in FEATURE_COLS],
            [builtins.round(100 * r[f"nz_{c}"] / n, 2) for c in FEATURE_COLS])

nn_all, nz_all = _fill(out)
nn_last, nz_last = _fill(out.filter(F.col("cohort_id") == latest))
display(pd.DataFrame({"feature": FEATURE_COLS,
                      "pct_filled_all_cohorts": nn_all, "pct_non_zero_all_cohorts": nz_all,
                      f"pct_filled_{latest}": nn_last, f"pct_non_zero_{latest}": nz_last}))

## Feature dictionary, business rules and business logic

**Features created (14)**. Panel = the HCP's Dx panel in the window; "no ESRD" = no ESRD / dialysis code for the patient in the window.

| # | Feature | Definition | Window |
| --- | --- | --- | --- |
| 1 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_T2D_CKD_IN_LAST_365_DAYS_DX` | Panel patients with T2D and CKD, no ESRD | 12 months |
| 2 | `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_T2D_CKD_IN_LAST_365_DAYS_DX` | #1 / panel size | 12 months |
| 3 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_HF_LVEF_GE_40_PROXY_IN_LAST_365_DAYS_DX` | Panel patients with an `I50.3x` code and no `I50.2x` code | 12 months |
| 4 | `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_HF_LVEF_GE_40_PROXY_IN_LAST_365_DAYS_DX` | #3 / panel size | 12 months |
| 5 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_CKD_AND_HF_IN_LAST_365_DAYS_DX` | Panel patients with CKD and any HF (`I50*`), no ESRD | 12 months |
| 6 | `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_CKD_AND_HF_IN_LAST_365_DAYS_DX` | #5 / panel size | 12 months |
| 7 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_ANY_KERENDIA_INDICATION_IN_LAST_365_DAYS_DX` | Panel patients with T2D+CKD, T1D+CKD or the HF proxy | 12 months |
| 8 | `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_ANY_KERENDIA_INDICATION_IN_LAST_365_DAYS_DX` | #7 / panel size | 12 months |
| 9 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_T1D_CKD_IN_LAST_365_DAYS_DX` | Panel patients with T1D and CKD, no ESRD (count only) | 12 months |
| 10 | `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_CKD_STAGE_3_IN_LAST_365_DAYS_DX` | Share of the panel with a stage 3 code | 12 months |
| 11 | `PCT_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_CKD_STAGE_4_IN_LAST_365_DAYS_DX` | Share of the panel with a stage 4 code | 12 months |
| 12 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_DIABETIC_CKD_IN_LAST_730_DAYS_DX` | 24-month panel patients with (T2D or T1D) and CKD, no ESRD (replaces the 3.0 `DB_KD` running total) | 24 months |
| 13 | `NUM_OF_PATIENTS_IN_HCP_DX_PANEL_WITH_CKD_IN_LAST_730_DAYS_DX` | 24-month panel patients with CKD, no ESRD (replaces the 3.0 `KD` running total) | 24 months |
| 14 | `LOGRATIO_PATIENTS_ANY_KERENDIA_INDICATION_3M` | `ln((n_3M/3 + 0.5) / ((n_12M - n_3M)/9 + 0.5))`, n = panel patients with any Kerendia indication | 3 vs 12 months |

**Business rules**

1. **ICD-10 code groups:** T2D `E11*`; T1D `E10*`; HF `I50*`; CKD (non-ESRD) `N18.1`-`N18.5` (incl. `N18.30/31/32`), `N18.9`, `E11.21/22/29`, `E10.21/22/29`, `I12.9`, `I13.0`, `I13.10`; ESRD / dialysis `N18.6`, `Z99.2`, `I12.0`, `I13.11`, `I13.2`; stage 3 `N18.3`, `N18.30-32`; stage 4 `N18.4`.
2. **ESRD exclusion:** T2D+CKD, T1D+CKD, CKD+HF, diabetic CKD and CKD counts exclude patients with an ESRD / dialysis code in the window (Kerendia is not indicated in ESRD). Stage shares do not apply the exclusion.
3. **HF LVEF >= 40% proxy:** `I50.3x` (diastolic) and no `I50.2x` (systolic) in the window. Pending Medical sign-off (V3).
4. **Dx panel:** distinct patients with any Dx event-list claim where the HCP is the referring doctor, else the rendering doctor.
5. **Patient flags from any provider:** a patient's diagnoses come from all claims in the window, not only the HCP's own claims.
6. **No future data:** each cohort uses claims up to the end of its `lookback_end_date` month. Windows are calendar months (12 months = "365 days", 24 months = "730 days").
7. **Null policy:** `NUM_*` = 0 when nothing qualifies. `PCT_*` and the log-ratio are empty when the HCP has no 12-month panel.
8. **Universe:** every HCP in `HCP_Universe_without_call_hcps` (294,727) gets one row per cohort (28 claims cohorts).

**Business logic (steps)**

1. Read `dx_input_claims_all_events_features_input` (NBA 3.1 Dx event list, ICD-10 `DIAGNOSIS_CODE`, 2021-01-02 onwards), set `HCP = COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID)` and the claim month.
2. Build the panel table (HCP x patient x month) for universe HCPs.
3. Build patient x month code flags (T2D, T1D, CKD, ESRD, stage 3, stage 4, HF, I50.2x, I50.3x) from claims by any provider, for patients that appear in a universe panel.
4. For each cohort: combine patient flags over the 12-month and 24-month windows ending with the lookback-end month, derive the indication flags, join them to the HCP's 3, 12 and 24-month panels and count distinct patients.
5. Turn the counts into the 14 features (shares divide by the 12-month panel size).
6. Cross-join the universe with all cohorts, fill `NUM_*` with 0, add `HCP_COHORT_ID = PROVIDER_ID_cohort_id` and write Delta to `hcp_features/claim_comorbidity`.
7. Validate: row count = HCPs x cohorts, unique keys, shares in [0, 1], counts not negative, any-indication >= each component, and a direct recompute of feature #1 from raw claims for 200 HCPs.